In [1]:
import warnings
warnings.filterwarnings('ignore')
import polars as pl
import polars.selectors as cs
import numpy as np
import math
import seaborn as sns
import os 
import re
import matplotlib.pyplot as plt
import gzip
import matplotlib.colors as mcolors
from scipy import stats
pl.Config.set_fmt_str_lengths(50)
pl.Config().set_tbl_rows(2000)
sns.set_style(style='white')
warnings.filterwarnings('ignore')

## Filtering

Filtering on:
- both parts at least 200 bp
- CRE is not an encode PLS
- promoter contains TSS


In [2]:
cd = "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/"
data = pl.read_csv(os.path.join(cd,"results/MPRA_analysis/CMPRA5/labeled_data_promoteroa_OA.tsv"), separator="\t")
data = data.filter(pl.col("right_bin").str.contains("null").not_()).rename({"OE": "CRE", "nr_reads": "nr_barcodes", "dist": "distance", "interaction": "ENCODE_labels"})
data = data.filter(pl.col("label") != "other - other")
data = data.filter(pl.col("any_tss") == "yes")
data = data.with_columns(
	CRE_length = pl.col("CRE").str.split("-").list.get(2).cast(pl.Int64) - pl.col("CRE").str.split("-").list.get(1).cast(pl.Int64),
	promoter_length = pl.col("promoter").str.split("-").list.get(2).cast(pl.Int64) - pl.col("promoter").str.split("-").list.get(1).cast(pl.Int64)
)
data = data.filter((pl.col("CRE_length") >= 100) & (pl.col("promoter_length") >= 100))
data = data.with_columns(pl.when(pl.all_horizontal(pl.any_horizontal(cs.matches("screen").str.contains("PLS")) & pl.any_horizontal(cs.matches("screen").is_null())))
			 			.then(pl.lit("PLS - undefined"))
						.when(pl.all_horizontal(pl.any_horizontal(cs.matches("screen").str.contains("ELS")) & pl.any_horizontal(cs.matches("screen").is_null())))
						.then(pl.lit("ELS - undefined"))
						 .when(pl.all_horizontal(cs.matches("screen").is_null())).then(pl.lit("undefined")).otherwise(pl.col("ENCODE_labels"))
						 .alias("ENCODE_labels"))
data = data.filter(pl.col("ENCODE_labels").str.contains("PLS - PLS").not_())

## Enhancers and silencers

In [112]:
silencers = data.sort("z_score").head(30).select(~cs.matches("left|right|tss|Val|std"))
enhancers = data.sort("z_score", descending=True).head(30).select(~cs.matches("left|right|tss|Val|std"))

## Multi interacting CREs

In [27]:
multi_prom = data.filter(pl.col("promoter").n_unique().over("CRE") >= 2) \
	.filter((pl.col("z_score").max().over("CRE") > 1.5) | (pl.col("z_score").min().over("CRE") < -1.5)) \
	.with_columns(activity_difference = np.abs(pl.col("z_score").max().over("CRE") - pl.col("z_score").min().over("CRE")))\
		.select(~cs.matches("left|right|tss|Val|std")).sort("activity_difference", descending=True)

multi_prom = multi_prom.filter(pl.col("CRE").is_in(multi_prom.select("CRE").unique(maintain_order=True).head(10)["CRE"]))

## Getting bin order of original sequences for enhancers and silencers

In [ ]:
!zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/mprasnakeflow/results/assigned_bcs_part1and2.tsv.gz' | cut -f -2,4,6,8,10,12,14 | awk 'NR > 1 && NF >=5' | cut -f 2 > temp.seqids.tsv

In [126]:
pl.concat([silencers, enhancers]).select(pl.col("CRE")).write_csv("CREs.temp.ids.tsv", include_header=False)
pl.concat([silencers, enhancers]).select(pl.col("promoter")).write_csv("promoter.temp.ids.tsv", include_header=False)

In [127]:
seq_ids = "temp.seqids.tsv"
cre_ids = "CREs.temp.ids.tsv"
prom_ids = "promoter.temp.ids.tsv"

In [128]:
!awk -v FS="\t" '{print $1"-"$2"-"$3"-"$4"\t"$5"-"$6"-"$7"-"$8"\t"$9}' \
	<(zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/promoteroabinaware_bins_all.filt_OA.bed.gz') \
		| grep -v "\-\-\-" | grep -Fwf CREs.temp.ids.tsv | grep -Fwf promoter.temp.ids.tsv | grep -Fwf <(sed 's/>//' temp.seqids.tsv) \
			> temp.actualbins.tsv

In [129]:
# Check uniqueness
! echo $(cut -f -2 temp.actualbins.tsv | sort | uniq |  wc -l) $(wc -l < temp.actualbins.tsv)
! echo $(wc -l < CREs.temp.ids.tsv) # looks like there are some incorrect bins

72 72
60


In [130]:
actual_bins = pl.read_csv("temp.actualbins.tsv", separator="\t", has_header=False).rename({"column_1": "left_bin", "column_2": "right_bin"}) 

In [131]:
actual_bins.with_columns(column_3 = pl.col("column_3").str.split(",").list.len()).head()

left_bin,right_bin,column_3
str,str,u32
"""chr1-44988261-44988729-.""","""chr1-45012148-45012412-+""",23
"""chr1-45012148-45012412-+""","""chr1-44988261-44988729-.""",21
"""chr1-45314479-45314700-.""","""chr1-45012148-45012412-+""",12
"""chr1-149474939-149475509-+""","""chr1-149498752-149498959-.""",10
"""chr10-122560249-122560775-+""","""chr10-122479620-122479771-.""",11


In [132]:
! grep "chr1-44988261-44988729-." temp.actualbins.tsv

chr1-44988261-44988729-.	chr1-45012148-45012412-+	m84066_240516_024507_s2/116200304/ccs,m84066_240516_024507_s2/117968625/ccs,m84066_240516_024507_s2/119738712/ccs,m84066_240516_024507_s2/123341587/ccs,m84066_240516_024507_s2/125175647/ccs,m84066_240516_024507_s2/134353403/ccs,m84066_240516_024507_s2/161482149/ccs,m84066_240516_024507_s2/197003724/ccs,m84066_240516_024507_s2/236719141/ccs,m84066_240516_024507_s2/264769070/ccs,m84066_240516_024507_s2/53546331/ccs,m84066_240516_024507_s2/67899501/ccs,m84066_240516_024507_s2/69010563/ccs,m84066_240623_102831_s3/101520839/ccs,m84066_240623_102831_s3/146478915/ccs,m84066_240623_102831_s3/157025067/ccs,m84066_240623_102831_s3/169610701/ccs,m84066_240623_102831_s3/239473830/ccs,m84066_240623_102831_s3/83168331/ccs,m84066_240623_122801_s1/17567107/ccs,m84066_240623_122801_s1/247466350/ccs,m84066_240623_122801_s1/37097751/ccs,m84066_240623_122801_s1/37687676/ccs
chr1-45012148-45012412-+	chr1-44988261-44988729-.	m84066_240516_024507_s2/34406993/

In [133]:
# Get the silencers with the bin order in which they were sequenced, and additionally in a CRE - promoter order
silencers_with_bin_order = silencers.select(pl.col("CRE").alias("left_bin"), pl.col("promoter").alias("right_bin"), pl.all())
silencers_with_bin_order = pl.concat([silencers_with_bin_order, 
									  actual_bins.select(pl.exclude("column_3")).join(silencers, left_on=["left_bin", "right_bin"], right_on=["promoter", "CRE"], coalesce = False)])

enhancers_with_bin_order = enhancers.select(pl.col("CRE").alias("left_bin"), pl.col("promoter").alias("right_bin"), pl.all())
enhancers_with_bin_order = pl.concat([enhancers_with_bin_order, 
									  actual_bins.select(pl.exclude("column_3")).join(enhancers, left_on=["left_bin", "right_bin"], right_on=["promoter", "CRE"], coalesce = False)])
enhancers_with_bin_order.height


51

## Getting bin order of original sequences for multi-promoter CREs

In [29]:
multi_prom.select(pl.col("CRE")).write_csv("CREs.multiprom.temp.ids.tsv", include_header=False)
multi_prom.select(pl.col("promoter")).write_csv("promoter.multiprom.temp.ids.tsv", include_header=False)
cre_ids = "CREs.multiprom.temp.ids.tsv"
prom_ids = "promoter.multiprom.temp.ids.tsv"

In [30]:
!awk -v FS="\t" '{print $1"-"$2"-"$3"-"$4"\t"$5"-"$6"-"$7"-"$8"\t"$9}' \
	<(zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/promoteroabinaware_bins_all.filt_OA.bed.gz') \
		| grep -v "\-\-\-" | grep -Fwf CREs.multiprom.temp.ids.tsv | grep -Fwf promoter.multiprom.temp.ids.tsv | grep -Fwf <(sed 's/>//' temp.seqids.tsv) \
			> temp.multiprom.actualbins.tsv

In [32]:
actual_bins = pl.read_csv("temp.multiprom.actualbins.tsv", separator="\t", has_header=False).rename({"column_1": "left_bin", "column_2": "right_bin"})

In [34]:
actual_bins.with_columns(column_3 = pl.col("column_3").str.split(",").list.len()).head()

left_bin,right_bin,column_3
str,str,u32
"""chr14-20688098-20688455-+""","""chr14-20955217-20955500-.""",16
"""chr14-20955217-20955500-.""","""chr14-20890827-20891165-+""",16
"""chr14-35115858-35116161-.""","""chr14-35121748-35122080--""",70
"""chr14-35121748-35122080--""","""chr14-35115858-35116161-.""",67
"""chr14-35122081-35122903--""","""chr14-35115858-35116161-.""",41


Dont forget: there are more sequences in the bin, that are not used in the end, so which are not counted in nr_seqs

In [36]:
multi_prom.filter(pl.col("CRE") == "chr14-20955217-20955500-.")

logFC,nr_barcodes,nr_seqs,label,ENCODE_labels,distance,target_genes,effect,promoter,CRE,promoter_only,z_score,CRE_length,promoter_length,activity_difference
f64,i64,i64,str,str,f64,str,str,str,str,f64,f64,i64,i64,f64
-1.086703,11,2,"""target - other""","""PLS - undefined""",267082.0,"""ANG""","""no effect""","""chr14-20688098-20688455-+""","""chr14-20955217-20955500-.""",-1.379707,0.916286,283,357,4.60108
0.618555,11,1,"""negative - other""","""PLS - undefined""",64362.5,"""RNASE3""","""upregulating""","""chr14-20890827-20891165-+""","""chr14-20955217-20955500-.""",-1.287388,5.517367,283,338,4.60108


In [38]:
multi_prom_with_binorder = multi_prom.select(pl.col("CRE").alias("left_bin"), pl.col("promoter").alias("right_bin"), pl.all())
multi_prom_with_binorder = pl.concat([multi_prom_with_binorder, 
									  actual_bins.select(pl.exclude("column_3")).join(multi_prom, left_on=["left_bin", "right_bin"], right_on=["promoter", "CRE"], coalesce = False)])
multi_prom_with_binorder.height, multi_prom.height

(37, 22)

## Orientation of CREs

In [89]:
!zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/singlebinssmalleroverlap_left_bins_OA.bed.gz' \
	| awk -v FS="\t" '{print $1"-"$2"-"$3"-.\t"$4"\t"$5}' | grep -Fwf CREs.temp.ids.tsv | \
		grep -Fwf <(cut -f 3 temp.actualbins.tsv | sed 's/,/\n/g') > left_bins_with_orientation.temp.tsv

!zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/singlebinssmalleroverlap_right_bins_OA.bed.gz' \
	| awk -v FS="\t" '{print $1"-"$2"-"$3"-.\t"$4"\t"$5}' | grep -Fwf CREs.temp.ids.tsv | \
		grep -Fwf <(cut -f 3 temp.actualbins.tsv | sed 's/,/\n/g') > right_bins_with_orientation.temp.tsv


In [ ]:
!paste <(zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/singlebinssmalleroverlap_left_bins_OA.bed.gz' \
	| awk -v FS="\t" '{print $1"-"$2"-"$3,$5}' )\
		<(zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/singlebinssmalleroverlap_right_bins_OA.bed.gz' \
	| awk -v FS="\t" '{print $1"-"$2"-"$3,$5}')| awk -v OFS="\t" '{print $1,$3,$2,$4}' | \
		grep -Fwf <(cut -f -2 temp.actualbins.tsv | sed 's/-\.//g' | sed 's/-+//g' | sed 's/--//g') | sort | uniq \
			> temp.actualbins.withorientation.tsv

In [ ]:
# investigate if majority are in one or two orientations
!cut -f 1,3 right_bins_with_orientation.temp.tsv | sort | uniq -c 

      8 chr1-149390448-149390792-.	+
      6 chr1-149390448-149390792-.	-
      7 chr1-149437428-149437700-.	+
      7 chr1-149498752-149498959-.	+
      3 chr1-149498752-149498959-.	-
      8 chr1-44988261-44988729-.	+
     13 chr1-44988261-44988729-.	-
     15 chr10-122546197-122546461-.	+
      3 chr10-122546197-122546461-.	-
     16 chr10-28001347-28001579-.	+
     18 chr11-115698711-115698946-.	-
     15 chr12-8687887-8688146-.	-
      4 chr12-92694400-92694607-.	+
     16 chr12-92694400-92694607-.	-
      7 chr14-20860518-20860752-.	+
      9 chr14-20860518-20860752-.	-
     28 chr14-20910989-20911246-.	+
     10 chr14-20910989-20911246-.	-
      8 chr14-20971623-20972006-.	+
     23 chr14-23028094-23028833-.	+
      7 chr14-96323573-96323971-.	-
      5 chr16-82042949-82043267-.	+
      8 chr16-82042949-82043267-.	-
     10 chr17-39846816-39847278-.	+
      5 chr17-75127197-75127537-.	-
     30 chr2-207487935-207488135-.	+
      8 chr2-39416724-39417440-.	+
     12 chr2-73880112

In [109]:
!cut -f 1,3 left_bins_with_orientation.temp.tsv | sort | uniq | sed 's/\s\+/\t/g' | cut -f 1  | sort | uniq -c | awk '{print $1}' | sort | uniq -c

     20 1
     20 2


In [70]:
enhancers.height

30

## Write to files

In [41]:
#silencers_with_bin_order.write_csv(os.path.join(cd,"results/luciferase_design/luciferase_design_silencers.tsv"), separator="\t")
#enhancers_with_bin_order.write_csv(os.path.join(cd,"results/luciferase_design/luciferase_design_enhancers.tsv"), separator="\t")
multi_prom_with_binorder.write_csv(os.path.join(cd,"results/luciferase_design/luciferase_design_dual_function.tsv"), separator="\t")